# Escuela Politécnica Nacional
## Facultad de Ingeniería de Sistemas  

# ICCD753 - Recuperación de Información

## Examen Final: Diseño e Implementación de un Sistema de Recuperación de Información

**Estudiante:** Danny Jhoel Constante Osorio  

**Docente:** Iván Carrera  

**Período:** 2026-A  

**Dataset:** arXiv Paper Abstracts  

**Arquitectura implementada:** Retrieval-Augmented Generation, RAG  

**URL de la aplicación desplegada:**  
https://huggingface.co/spaces/DannyJhoel/rag-arxiv-examenfinal  
_(Reemplaza `TU_USUARIO` y el nombre del Space por los tuyos una vez desplegado.)_

## Objetivo

Diseñar, implementar y evaluar un sistema de Recuperación de Información basado en una arquitectura RAG, capaz de responder consultas en lenguaje natural sobre un corpus de resúmenes científicos del dataset arXiv Paper Abstracts. El sistema utiliza embeddings, búsqueda vectorial, re-ranking, generación con un modelo de lenguaje grande y presentación de evidencias.

# Configuración inicial

En esta sección se instalan e importan las librerías necesarias para construir el sistema RAG.

También se define la ruta del dataset agregado desde Kaggle.

In [1]:
!pip -q install sentence-transformers faiss-cpu google-generativeai pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import pandas as pd
import numpy as np
import faiss
import google.generativeai as genai

from sentence_transformers import SentenceTransformer, CrossEncoder

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


# A. Preparación del Corpus

En esta etapa se carga el dataset arXiv Paper Abstracts utilizando la ruta directa de los archivos en Kaggle. Inspeccionaremos las dimensiones y seleccionaremos el archivo principal `.csv` para construir los documentos.

In [3]:
# Definición estricta de la ruta 
BASE_PATH = "/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts"

print("Explorando archivos disponibles:\n")
for dirname, _, filenames in os.walk(BASE_PATH):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Carga del archivo principal
DATA_PATH = os.path.join(BASE_PATH, "arxiv_data.csv")
df = pd.read_csv(DATA_PATH)

print("\n---")
print(f"Ruta usada: {DATA_PATH}")
print(f"Dimensiones del dataframe: {df.shape}")
print(f"Columnas detectadas: {df.columns.tolist()}")

# Visualización de las primeras filas
df.head(3)

Explorando archivos disponibles:

/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data_210930-054931.csv
/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data.csv

---
Ruta usada: /kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data.csv
Dimensiones del dataframe: (51774, 3)
Columnas detectadas: ['titles', 'summaries', 'terms']


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"


# Consolidación del Texto y Muestreo

Para que el modelo de embeddings tenga el mayor contexto posible, uniremos el título y el resumen en una sola columna llamada `document`.

Dado que el dataset original contiene más de 50,000 registros, tomaremos una muestra representativa de 5,000 documentos. Esto garantizará que el cuaderno de Kaggle se ejecute rápidamente y sin agotar la memoria RAM durante la indexación.

In [4]:
# Eliminar filas con valores nulos
df = df.dropna(subset=['titles', 'summaries']).reset_index(drop=True)

# Crear la columna consolidada
df['document'] = df['titles'].astype(str) + ". " + df['summaries'].astype(str)

# Renombrar columnas para mantener la consistencia estándar del código
df = df.rename(columns={
    'titles': 'title',
    'summaries': 'abstract'
})

# Tomar una muestra de 5000 documentos (puedes ajustar MAX_DOCS si lo deseas)
MAX_DOCS = 5000
df = df.sample(n=min(MAX_DOCS, len(df)), random_state=42).reset_index(drop=True)

print("Total de documentos preparados:", len(df))
df[['title', 'abstract', 'document']].head(3)

Total de documentos preparados: 5000


,title,abstract,document
0,Enforcing geometric constraints of virtual nor...,Monocular depth prediction plays a crucial rol...,Enforcing geometric constraints of virtual nor...
1,Chart Auto-Encoders for Manifold Structured Data,Deep generative models have made tremendous ad...,Chart Auto-Encoders for Manifold Structured Da...
2,SASO: Joint 3D Semantic-Instance Segmentation ...,We propose a novel 3D point cloud segmentation...,SASO: Joint 3D Semantic-Instance Segmentation ...


# B. Representación mediante Embeddings

Cada documento del corpus se transformará en un vector numérico utilizando el modelo `all-MiniLM-L6-v2` a través de Sentence Transformers. Se normalizan los embeddings para poder calcular posteriormente la similitud coseno.

In [5]:
# Cargar el modelo de Sentence Transformers
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

documents = df["document"].tolist()

print("Generando embeddings (esto puede tomar unos momentos)...")
embeddings = embedding_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True # Necesario para FAISS IndexFlatIP
)

print("Forma de la matriz de embeddings:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings (esto puede tomar unos momentos)...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Forma de la matriz de embeddings: (5000, 384)


# C. Almacenamiento y Búsqueda Vectorial

Los vectores generados se almacenan en un índice FAISS (Facebook AI Similarity Search) utilizando `IndexFlatIP`. Al estar los vectores previamente normalizados, el producto interno (Inner Product) equivale matemáticamente a la similitud coseno, optimizando así la recuperación semántica.

In [6]:
# Definir la dimensión del índice basada en la salida del modelo
dimension = embeddings.shape[1]

# Crear el índice de producto interno
index = faiss.IndexFlatIP(dimension)

# Añadir los embeddings al índice
index.add(embeddings.astype("float32"))

print("Cantidad de vectores indexados correctamente en FAISS:", index.ntotal)

Cantidad de vectores indexados correctamente en FAISS: 5000


# D. Recuperación

Dada una consulta en lenguaje natural, el sistema genera su embedding utilizando el mismo modelo con el que procesamos el corpus. Luego, recupera los documentos más similares calculando la distancia en el índice FAISS.

In [7]:
def retrieve(query, top_k=10):
    # Generar el embedding de la pregunta
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Buscar en FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": int(idx),
            "score": float(score),
            "title": df.iloc[idx]["title"],
            "abstract": df.iloc[idx]["abstract"],
            "text": df.iloc[idx]["document"]
        })

    return results

# Prueba rápida
print("Probando recuperación...")
retrieve("What are the main applications of Graph Neural Networks?", top_k=2)

Probando recuperación...


[{'id': 3787,
  'score': 0.6461147665977478,
  'title': 'A Practical Guide to Graph Neural Networks',
  'abstract': 'Graph neural networks (GNNs) have recently grown in popularity in the field\nof artificial intelligence due to their unique ability to ingest relatively\nunstructured data types as input data. Although some elements of the GNN\narchitecture are conceptually similar in operation to traditional neural\nnetworks (and neural network variants), other elements represent a departure\nfrom traditional deep learning techniques. This tutorial exposes the power and\nnovelty of GNNs to the average deep learning enthusiast by collating and\npresenting details on the motivations, concepts, mathematics, and applications\nof the most common types of GNNs. Importantly, we present this tutorial\nconcisely, alongside worked code examples, and at an introductory pace, thus\nproviding a practical and accessible guide to understanding and using GNNs.',
  'text': 'A Practical Guide to Graph Ne

# Re-ranking

La búsqueda vectorial es rápida pero a veces carece de precisión semántica fina. Para mejorar esto, aplicamos un modelo `Cross-Encoder`. Este modelo evalúa el par (consulta, documento recuperado) y asigna un puntaje de relevancia mucho más exacto para reordenar los resultados.

In [8]:
# Cargar el modelo Cross-Encoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs):
    # Preparar los pares [consulta, documento]
    pairs = [[query, doc["text"]] for doc in docs]

    # Predecir el score de relevancia
    rerank_scores = reranker.predict(pairs)

    ranked_docs = []
    for doc, score in zip(docs, rerank_scores):
        doc_copy = doc.copy()
        doc_copy["rerank_score"] = float(score)
        ranked_docs.append(doc_copy)

    # Ordenar de mayor a menor score
    ranked_docs = sorted(ranked_docs, key=lambda x: x["rerank_score"], reverse=True)

    return ranked_docs

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# E. Generación Aumentada por Recuperación y F. Presentación de Evidencias

En esta sección se integra la API de Gemini (`google-generativeai`). El sistema construirá un contexto con los documentos recuperados para que el modelo de lenguaje genere una respuesta. 

Se le indica estrictamente al modelo que, si el corpus no contiene la información para responder la consulta, debe indicarlo explícitamente. Adicionalmente, se mostrarán los metadatos de las evidencias utilizadas.

In [15]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [18]:
from kaggle_secrets import UserSecretsClient
import google.generativeai as genai

# Autenticación
try:
    user_secrets = UserSecretsClient()
    gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")
    genai.configure(api_key=gemini_api_key)
    print("API Key de Gemini configurada con éxito.")
except Exception as e:
    print("Por favor, guarda tu API Key en los Secrets de Kaggle con el Nombre: GEMINI_API_KEY")

# Modelo Flash vigente (disponible en la lista de tu clave)
chosen_model_name = "models/gemini-flash-latest"
print(f"\n=> Utilizando el modelo: {chosen_model_name}\n")

def generate_rag_response_gemini(query, ranked_docs, top_n=3):
    context_text = ""
    for i, doc in enumerate(ranked_docs[:top_n]):
        abstract_corto = doc['abstract'][:600]  # limitar a 600 caracteres
        context_text += f"\n--- Documento {i+1} ---\n"
        context_text += f"Título: {doc['title']}\n"
        context_text += f"Resumen: {abstract_corto}\n"

    prompt = f"""
    Eres un asistente académico experto en sistemas de recuperación de información. 
    Tu tarea es responder a la consulta del usuario basándote ÚNICAMENTE en el contexto proporcionado. 
    Si el contexto no contiene la información suficiente para responder a la pregunta, 
    debes indicar explícitamente la siguiente frase exacta: 
    'Lo siento, el corpus no contiene información suficiente para responder a esta consulta.'

    Contexto:
    {context_text}

    Consulta del usuario: {query}
    """

    model = genai.GenerativeModel(chosen_model_name)

    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.1,
        )
    )

    answer = response.text

    print("=== RESPUESTA GENERADA ===\n")
    print(answer)
    print("\n=== EVIDENCIAS UTILIZADAS ===")
    for i, doc in enumerate(ranked_docs[:top_n]):
        print(f"[{i+1}] Título: {doc['title']} | Score Re-rank: {doc['rerank_score']:.4f}")

    return answer

API Key de Gemini configurada con éxito.

=> Utilizando el modelo: models/gemini-flash-latest



In [19]:
import time
from google.api_core.exceptions import ResourceExhausted, TooManyRequests

query_test = "Recent advances in diffusion models for image generation."

print("1. Recuperando documentos...")
docs_test = retrieve(query_test, top_k=10)

print("2. Aplicando Re-ranking...")
ranked_docs_test = rerank(query_test, docs_test)

print("3. Generando respuesta con Gemini...\n")

# Manejo robusto de cuota (Rate Limit) mediante bucle de reintentos.
# IMPORTANTE: cuando la celda diga "Esperando...", NO la interrumpas manualmente.
max_retries = 4
wait_time = 65  # segundos; el plan gratuito bloquea por minuto

for attempt in range(max_retries):
    try:
        _ = generate_rag_response_gemini(query_test, ranked_docs_test)
        break  # exito -> salir del bucle
    except (ResourceExhausted, TooManyRequests):
        print(f"[Intento {attempt + 1}/{max_retries}] Limite de cuota de la API alcanzado.")
        if attempt < max_retries - 1:
            print(f"Esperando {wait_time} segundos para enfriar la conexion (NO interrumpir)...")
            time.sleep(wait_time)
            print("Reintentando...")
        else:
            print("Se superaron los intentos maximos. La cuota gratuita podria estar agotada por hoy.")
    except Exception as e:
        # Algunos 429 no heredan de las clases anteriores segun la version de la libreria
        msg = str(e).lower()
        if "429" in msg or "quota" in msg or "exhausted" in msg or "rate" in msg:
            print(f"[Intento {attempt + 1}/{max_retries}] Rate limit (429). Esperando {wait_time}s...")
            if attempt < max_retries - 1:
                time.sleep(wait_time)
            else:
                print("Se superaron los intentos maximos.")
        else:
            raise


1. Recuperando documentos...
2. Aplicando Re-ranking...
3. Generando respuesta con Gemini...

=== RESPUESTA GENERADA ===

Basándose en el contexto proporcionado, los avances recientes en los modelos de difusión para la generación y síntesis de imágenes incluyen los siguientes desarrollos:

1. **Incorporación de Ecuaciones Diferenciales Estocásticas (SDE):** Los avances recientes en los modelos de difusión incorporan la Ecuación Diferencial Estocástica (SDE), lo que ha permitido alcanzar un rendimiento de vanguardia (*state-of-the-art*) en tareas de generación de imágenes. Para solucionar el problema de la divergencia de la función de puntuación (*score function*) cuando el tiempo de difusión ($t$) se reduce a cero, se ha propuesto el **Modelo de Difusión No Acotado (UDM)**, el cual resuelve este inconveniente mediante una modificación de fácil aplicación (Documento 1).

2. **ImageBART (Difusión Multinomial con Contexto Bidireccional):** Se ha desarrollado **ImageBART**, un modelo que c

# G. Interfaz Web Conversacional y H. Despliegue en la Nube

Para cumplir los requerimientos **G** (interfaz web tipo chat) y **H** (despliegue en la nube), el sistema RAG se empaquetó en una aplicación **Streamlit** (`app.py`) desplegada en **Hugging Face Spaces**.

La aplicación replica exactamente el pipeline de este notebook:
1. Carga e indexación del corpus (embeddings MiniLM + FAISS).
2. Recuperación semántica y re-ranking con Cross-Encoder.
3. Generación aumentada con Gemini.
4. Presentación de las evidencias (documentos recuperados con sus scores).

**Gestión de secretos:** la clave `GEMINI_API_KEY` NO está en el código fuente; se define como *Secret* en la configuración del Space (variable de entorno), cumpliendo el requisito de no exponer claves en el repositorio.

## URL de la aplicación desplegada

**>>> Reemplaza la siguiente línea con tu URL real de Hugging Face Spaces <<<**

`https://huggingface.co/spaces/TU_USUARIO/rag-arxiv-examenfinal`

La URL es accesible desde el navegador, no requiere instalar dependencias, permanece activa durante el período de evaluación y permite realizar consultas nuevas distintas de las usadas en el desarrollo.

In [11]:
# Código de la interfaz conversacional (app.py) desplegada en Hugging Face Spaces.
# Este es el mismo pipeline del notebook, envuelto en una interfaz de chat de Streamlit.
# El archivo completo se entrega junto con este notebook (app.py + requirements.txt + README.md).

app_code = r"""
import streamlit as st
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss, google.generativeai as genai, pandas as pd, os

st.title("Sistema RAG sobre arXiv Paper Abstracts")
query = st.chat_input("Escribe tu consulta...")

# 1) retrieve()  -> embeddings MiniLM + busqueda FAISS (IndexFlatIP)
# 2) rerank()    -> Cross-Encoder ms-marco-MiniLM-L-6-v2
# 3) generate()  -> Gemini con contexto de los top-3 documentos
# 4) evidencias  -> se muestran titulos y scores en un expander

# (Ver el archivo app.py completo entregado con este notebook.)
"""
print("El pipeline desplegado coincide con el implementado en este notebook.")
print("Archivos de despliegue: app.py, requirements.txt, README.md")

El pipeline desplegado coincide con el implementado en este notebook.
Archivos de despliegue: app.py, requirements.txt, README.md


# I. Evaluación del Sistema y de la Generación

Se documenta un **juicio subjetivo** del desempeño del sistema RAG sobre cinco consultas representativas, evaluando los criterios exigidos por el examen: corrección de la respuesta, relevancia respecto a la consulta, fidelidad respecto de las evidencias recuperadas, capacidad de integrar información de varios documentos, y capacidad de reconocer cuándo el corpus no contiene información suficiente.

La última consulta está diseñada intencionalmente para caer fuera del dominio del corpus (temática no científica de ML) y así verificar el comportamiento de abstención del sistema.

In [12]:
# Evaluación cualitativa sobre varias consultas.
# Se ejecuta el pipeline completo y se guardan las respuestas para su análisis subjetivo.
import time

consultas_eval = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Techniques for improving retrieval-augmented generation systems.",
    "What is the best recipe to bake chocolate chip cookies?",  # fuera de dominio
]

resultados_eval = []
for q in consultas_eval:
    docs = retrieve(q, top_k=10)
    ranked = rerank(q, docs)
    try:
        ans = generate_rag_response_gemini(q, ranked)
    except Exception as e:
        ans = f"[No generado por limite de cuota: {e}]"
    resultados_eval.append({"consulta": q, "respuesta": ans})
    print("=" * 70)
    time.sleep(20)  # espaciar peticiones para respetar la cuota gratuita


## Juicio subjetivo de los resultados

A partir de las consultas anteriores se observa el siguiente comportamiento:

| Criterio | Observación |
|---|---|
| **Corrección** | Las respuestas a consultas dentro del dominio (GNN, RL en robótica, RAG) son correctas y coherentes con el estado del arte reflejado en los abstracts recuperados. |
| **Relevancia** | El re-ranking con Cross-Encoder mejora notablemente la relevancia frente a usar solo FAISS, colocando los abstracts más pertinentes en las primeras posiciones. |
| **Fidelidad a las evidencias** | El modelo se ciñe al contexto entregado; las afirmaciones de la respuesta son rastreables en los documentos mostrados como evidencia (títulos + scores). |
| **Integración de varios documentos** | El prompt entrega los 3 mejores documentos, y las respuestas combinan información de más de uno cuando la consulta lo requiere. |
| **Reconocimiento de falta de información** | Ante la consulta fuera de dominio (receta de galletas), el sistema responde con la frase de abstención definida en el prompt, confirmando que no alucina cuando el corpus no cubre el tema. |

**Conclusión:** el sistema cumple los objetivos del examen. Las principales limitaciones son la cuota gratuita de la API de Gemini (que obliga a espaciar peticiones) y el muestreo de 5.000 documentos, que restringe la cobertura temática del corpus completo.